## 3-way 비교: Centralized / Independent / 우리 방법론(Sparse Graph)

이 노트북 하나만 실행하면 됨 — Drive 마운트 필요 없음. 레포를 통째로 clone해서
`Code/`(agent.py, offer.py, localplan.py, models.py, config.py, utils.py, vlm.py,
tracker.py, universal_graph.py)와 `experiment/`(commander.py, independent.py)를
둘 다 `sys.path`에 올려서 세 방법론이 전부 같은 코드 소스에서 동작함.

`Code/agent.py`, `Code/vlm.py`는 GitHub 원본 그대로 두고, clone된 **로컬 사본만**
아래 "코드 패치" 셀에서 런타임에 고침 (원본 레포 파일은 건드리지 않음).

### 0. 셋업

In [ ]:
!git clone https://github.com/egggbeee-dev/kcc-journal-0817.git /content/kcc-journal-0817

import sys
REPO_DIR       = "/content/kcc-journal-0817"
DATA_DIR       = f"{REPO_DIR}/Data"
CODE_DIR       = f"{REPO_DIR}/Code"
EXPERIMENT_DIR = f"{REPO_DIR}/experiment"

for p in (CODE_DIR, EXPERIMENT_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

!find {REPO_DIR}/Data -maxdepth 3
!find {REPO_DIR}/Code -maxdepth 1
!find {REPO_DIR}/experiment -maxdepth 1

### 0-1. 코드 패치 (로컬 clone 사본만, GitHub 원본은 그대로)

레포 원본 `Code/agent.py`, `Code/vlm.py`에 아직 남아있는 버그 두 가지를 clone된
로컬 사본에만 패치함:

1. `agent.py` — 방마다 이미지 2~4장 제약이 있어서 지금 데이터(방마다 개수 다름)로는
   `build_agents()`가 `ValueError`를 던짐 → 이미지 1장 이상이면 그대로 쓰도록 완화
2. `vlm.py` — Qwen 백엔드일 때 토큰 카운팅이 없어서 tracker가 TC=0으로 찍음 → 지금은
   OpenAI 백엔드라 당장 안 걸리지만, 나중에 Qwen 테스트할 때 필요하니 미리 패치

In [ ]:
def _patch_once(path, old, new, label):
    with open(path, encoding="utf-8") as f:
        src = f.read()
    if new in src:
        print(f"  [PATCH] {label}: 이미 패치되어 있음 — 건너뜀")
        return
    if old not in src:
        print(f"  [PATCH][WARN] {label}: 원본 텍스트를 못 찾음 — 레포가 바뀌었을 수 있음, 수동 확인 필요")
        return
    src = src.replace(old, new)
    with open(path, "w", encoding="utf-8") as f:
        f.write(src)
    print(f"  [PATCH] {label}: 완료")


# 1) agent.py — 2~4장 이미지 제약 제거
_agent_old = '''    agents: List[Agent] = []
    for i, (agent_id, imgs) in enumerate(zone_image_map.items()):
        if not (MIN_ZONE_IMAGES <= len(imgs) <= MAX_ZONE_IMAGES):
            raise ValueError(
                f"{agent_id}: zone_images는 {MIN_ZONE_IMAGES}~{MAX_ZONE_IMAGES}장이어야 합니다. "
                f"받은 값: {len(imgs)}"
            )
        agents.append(Agent('''
_agent_new = '''    agents: List[Agent] = []
    for i, (agent_id, imgs) in enumerate(zone_image_map.items()):
        # 방마다 실제 이미지 개수를 그대로 사용 (2~4장 제약 없음)
        if len(imgs) < 1:
            raise ValueError(f"{agent_id}: zone_images가 비어 있습니다.")
        agents.append(Agent('''
_patch_once(f"{CODE_DIR}/agent.py", _agent_old, _agent_new, "agent.py (이미지 개수 제약)")


# 2) vlm.py — Qwen 백엔드 토큰 카운팅 추가
_vlm_path = f"{CODE_DIR}/vlm.py"
with open(_vlm_path, encoding="utf-8") as f:
    _vlm_src = f.read()

if "_record_local_usage" not in _vlm_src:
    _helper = (
        "def _record_local_usage(prompt_tokens: int, completion_tokens: int) -> None:\n"
        "    # 로컬(Qwen) 백엔드는 OpenAI처럼 usage를 안 돌려주므로 직접 토큰 길이를 세서 기록한다.\n"
        '    _last_usage["prompt_tokens"]     = prompt_tokens\n'
        '    _last_usage["completion_tokens"] = completion_tokens\n'
        "    with _usage_lock:\n"
        '        _total_usage["prompt_tokens"]     += prompt_tokens\n'
        '        _total_usage["completion_tokens"] += completion_tokens\n'
    )
    _marker = "def _load_qwen():"
    if _marker in _vlm_src:
        _vlm_src = _vlm_src.replace(_marker, _helper + "\n\n" + _marker, 1)
        _old_ret = (
            '    gen_ids  = out.sequences[:, inputs["input_ids"].shape[1]:]\n'
            "    text_out = _qwen_processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()\n"
            "\n"
            "    log_probs: List[float] = []\n"
            "    if return_logprobs and out.scores:\n"
            "        for step_idx, step_scores in enumerate(out.scores):\n"
            "            token_id = gen_ids[0, step_idx].item()\n"
            "            log_probs.append(\n"
            '                torch.log_softmax(step_scores[0], dim=-1)[token_id].item()\n'
            "            )\n"
            "\n"
            "    return text_out, log_probs"
        )
        _new_ret = _old_ret.replace(
            "    return text_out, log_probs",
            '    _record_local_usage(inputs["input_ids"].shape[1], gen_ids.shape[1])\n\n    return text_out, log_probs',
        )
        if _old_ret in _vlm_src:
            _vlm_src = _vlm_src.replace(_old_ret, _new_ret, 1)
            with open(_vlm_path, "w", encoding="utf-8") as f:
                f.write(_vlm_src)
            print("  [PATCH] vlm.py (Qwen 토큰 카운팅): 완료")
        else:
            print("  [PATCH][WARN] vlm.py: _run_qwen 반환부 패턴을 못 찾음 — 수동 확인 필요")
    else:
        print("  [PATCH][WARN] vlm.py: _load_qwen 마커를 못 찾음 — 수동 확인 필요")
else:
    print("  [PATCH] vlm.py (Qwen 토큰 카운팅): 이미 패치되어 있음 — 건너뜀")

In [ ]:
!pip install openai

In [ ]:
from google.colab import userdata
import os

os.environ["VLM_BACKEND"] = "openai"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import os, glob, json

def build_zone_image_map(room_selections: dict) -> dict:
    """
    room_selections: {zone_id: (room_type, floorplan_id)}
      - room_type: Data/Room/AI2THOR/ 아래 폴더명 그대로 ("Kitchen","Livingroom","Bathroom","Bedroom")
      - floorplan_id: 문자열 숫자, 예: "101"
    2개~8개(config.MAX_AGENTS) 아무 개수나 가능. 방마다 실제 이미지 개수 그대로 사용.
    """
    zone_image_map = {}
    for zone_id, (room_type, fp_id) in room_selections.items():
        folder = os.path.join(DATA_DIR, "Room", "AI2THOR", room_type)
        pattern = os.path.join(folder, f"floorplan_{fp_id}_*.png")
        imgs = sorted(glob.glob(pattern))
        if not imgs:
            print(f"  [WARN] {zone_id} ({room_type} floorplan_{fp_id}) 이미지 없음 — 경로/번호 확인 필요")
        zone_image_map[zone_id] = imgs
        print(f"  {zone_id} -> {len(imgs)}장: {[os.path.basename(p) for p in imgs]}")
    return zone_image_map


def load_tasks() -> list:
    with open(os.path.join(DATA_DIR, "Task", "tasks.json"), encoding="utf-8") as f:
        return json.load(f)

TASKS = load_tasks()
TASKS_BY_ID = {t["id"]: t for t in TASKS}

def get_task(task_id: str) -> str:
    return TASKS_BY_ID[task_id]["description"]

def get_ground_truth(task_id: str) -> dict:
    return TASKS_BY_ID[task_id]["ground_truth"]


SHARED_MODULE_NAMES = (
    "agent", "offer", "localplan", "models", "config",
    "utils", "vlm", "tracker", "universal_graph",
    "commander", "independent",
)

def reset_module_cache():
    """패치 직후나 commander.py/independent.py 재실행 시 캐시된 옛 버전이
    남지 않도록, 실행 셀 맨 앞에서 관련 모듈 캐시를 지운다."""
    for m in SHARED_MODULE_NAMES:
        if m in sys.modules:
            del sys.modules[m]

In [ ]:
# 세 방법론이 공유하는 태스크/방 구성 — 여기서 한 번만 지정
TASK_ID = "task_001"
TASK = get_task(TASK_ID)

ROOM_SELECTIONS = {
    "kitchen":     ("Kitchen",    "101"),
    "living_room": ("Livingroom", "226"),
    "bathroom":    ("Bathroom",   "415"),
    "bedroom":     ("Bedroom",    "303"),
}
zone_image_map = build_zone_image_map(ROOM_SELECTIONS)

### 1. Centralized 실행

agent0(commander)이 물리적 zone 없이 subordinate들의 관찰 보고서만 받아서 참여 agent
선별 + 지시(directive)를 단일 호출로 결정하고, subordinate는 그 지시만 따라 로컬
플랜을 만듦. 최종 정렬은 `universal_graph.merge_joint_plan`(Kahn's algorithm).

In [ ]:
reset_module_cache()

from agent import build_agents
from commander import run_centralized_commander
from tracker import tracker

METHOD = "Centralized"

subordinates = build_agents(zone_image_map)

tracker.start()
joint_plan_c, decisions_c, plans_c = run_centralized_commander(subordinates, TASK)
tracker.stop()

print(f"\n[{METHOD}] task={TASK_ID}")
print(tracker.summary(METHOD))

### 2. Independent 실행

각 agent가 `agent.py`의 self-assessment 참여 판단만 공유하고, 로컬 플랜은 자기 zone
이미지 + 태스크만 보고 완전히 독립적으로 세움. 다른 agent에게 뭔가를 넘기는 행동
자체는 할 수 있지만, 받는 쪽과 연결해줄 메커니즘이 없어서 고립된 스텝으로 남음.

In [ ]:
reset_module_cache()

from agent import build_agents
from independent import run_independent
from tracker import tracker

METHOD = "Independent"

agents = build_agents(zone_image_map)

tracker.start()
joint_plan_i, active_agents_i, plans_i = run_independent(agents, TASK)
tracker.stop()

print(f"\n[{METHOD}] task={TASK_ID}")
print(tracker.summary(METHOD))

### 3. 우리 방법론 실행

참여 판단(`agent.py`) → Offer 생성(`offer.py`) → Local Plan 생성(`localplan.py`) →
Auction 기반 매칭 + Graph 기반 충돌 검증/해소(`universal_graph.run`)까지 전 단계.

In [ ]:
reset_module_cache()

from agent import build_agents, decide_participation_all, filter_active_agents
from offer import generate_offers
from localplan import generate_local_plans
import universal_graph
from tracker import tracker

METHOD = "Ours (Sparse Graph)"

agents_g = build_agents(zone_image_map)

tracker.start()
decisions_g = decide_participation_all(agents_g, TASK)
active_agents_g = filter_active_agents(agents_g, decisions_g)
offers_g = generate_offers(active_agents_g, TASK)
plans_g = generate_local_plans(active_agents_g, offers_g, TASK)
result_g = universal_graph.run(active_agents_g, offers_g, plans_g, TASK)
tracker.stop()

joint_plan_g = result_g["joint_plan"]
print(result_g["joint_plan_text"])
print(f"\n[{METHOD}] task={TASK_ID}")
print(tracker.summary(METHOD))